In [ ]:
# Yet another RAG assistant for technical research

We'll show you how to construct a simple Retrieval Augmented Generation (RAG) pipeline (ie: LLMs + search) using a variety of free and open source tools you can mix and match. Featuring Intel Gaudi on Denvr Cloud.

This demo will be made publicly available in [github.com/denvrdata/denvrdemos](https://github.com/denvrdata/denvrdemos)

Takeaways:

1. Value of combining search with LLMs
2. Ease of applying custom RAG pipelines to existing solutions
3. Resilient solutions with OSS, Denvr and Gaudi

## Background

Over the past few years, tools like ChatGPT and Copilot have made LLMs a part of our daily lives, appearing in everything from email clients to code editors.
Despite their reach, LLMs still have notable limitations.
As probabilistic models trained on finite datasets, they exhibit the following weaknesses:

- **Overly General Responses** - Vague or non-specific answers due to broad training data.
- **Outdated Knowledge** - Training data is static, representing past information.
- **Lack of Verifiable Sources** - Responses lack citations, making it difficult to verify accuracy.
- **Hallucinations** - Occasionally generate plausible-sounding but incorrect or fabricated information.

One mitigation strategy is Retrieval Augmented Generation (RAG) pipelines which pairs LLMs with a search component and prompt engineering.
The diagram below covers one of the more common workflows, incorporating an embedding model and a vector database.
Conceptually, you just need a:

- **Document Store** - you can search with the input prompt
- **Query Augmentation Step** - to incorporate that relevant data/context into the original prompt you’re sending to the LLM.

![rag diagram](https://raw.githubusercontent.com/denvrdata/denvrdemos/refs/heads/main/yara/assets/images/RAG.drawio.svg)

In our diagram, we are:

1. Populating a vector database by:
   1. Parsing the raw documents (e.g., doc, pdf, html)
   2. Sending the raw text through a small embedding model
   3. Inserting the embedded text into the database
2. Running our prompt through the same embedding model to query the database for relevant stored context.
3. Augmenting the prompt with the most relevant context chunks and sending that new prompt to the LLM.

A nice feature of this approach is that we can experiment with different independent search and prompt augmentation strategies.
Similarly, you can also experiment with different LLMs, embedding models and reference datasets.

## Components

### LangChain

This is the primary framework we'll be using to glue our components together. This framework includes a diverse set of community contributed extensions which covers everthing from different vectorstores, document parsing, chunking strategies and model types.

[docs](https://python.langchain.com/docs/introduction/)

### DuckDB

We'll use duckdb to store our embedding vectors along with some mock application data. While dedicated vector database may be more performant, being able to simply store your embeddings as a simple table in a relational database is a great way to integrate these pipelines into existing applications. For this demo, the fact that duckdb is a simple local database (like sqlite) worked great, but you could also use extensions like PGVector in postgres.

### HuggingFace

We'll be downloading both our embedding and chat models from hugging face and wrapping our Gaudi text generation pipeline in langchains huggingface pipeline.

### Pytorch, Transformers and Optimum Habana

For our chat model we'll be using the standard pytorch and transformers libraries, but with a few Gaudi specifi changes using the habana frameworks.

NOTE: The optimum habana repo also includes a bunch of great examples to help folks get started

[examples](https://github.com/huggingface/optimum-habana/tree/main/examples)

### Gradio

We'll use a really simple chat interface to compare the base and rag output.

## Modules

For this demo, we'll be using two custom python modules.

1. yara.py - contains our platfrom agnosotic logic for managing duckdb tables, downloading files, building the pipeline RAG pipeline and launching a simple gradio UI.
2. gaudi.py - contains the gaudi specific logic for loading the chat LLM model on the gaudi HPUs

## Demo

Okay, enough background info. Let's play around with some code. We'll start by loading the yara module, creating our duckdb connection, and use the langchain `SitemapLoader` to download some raw html documents.

In [1]:
# This import is required only for jupyter notebooks, since they have their own eventloop
import nest_asyncio
nest_asyncio.apply()

import yara
conn = yara.connect()
yara.add_site(conn, "denvr", "https://docs.denvrdata.com/docs/sitemap.xml")
yara.download(conn)
conn.table('downloads')

USER_AGENT environment variable not set, consider setting it to identify your requests.
11/19/2024 06:15:49 - INFO - yara - Adding https://docs.denvrdata.com/docs/sitemap.xml to downloads table
11/19/2024 06:15:49 - INFO - yara - Downloading pages from https://docs.denvrdata.com/docs/sitemap.xml
Fetching pages: 100%|##########| 33/33 [00:12<00:00,  2.73it/s]
11/19/2024 06:16:03 - INFO - yara - Process https://docs.denvrdata.com/docs/
11/19/2024 06:16:03 - INFO - yara - Saving document to docs/denvr/index.json
11/19/2024 06:16:03 - INFO - yara - Process https://docs.denvrdata.com/docs/overview/getting-started
11/19/2024 06:16:03 - INFO - yara - Saving document to docs/denvr/overview__getting-started.json
11/19/2024 06:16:03 - INFO - yara - Process https://docs.denvrdata.com/docs/overview/getting-started/launch-a-virtual-machine
11/19/2024 06:16:03 - INFO - yara - Saving document to docs/denvr/overview__getting-started__launch-a-virtual-machine.json
11/19/2024 06:16:03 - INFO - yara - Pr

┌────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────┬──────────┐
│                                             source                                             │                                   destination                                    │ sitename │
│                                            varchar                                             │                                     varchar                                      │ varchar  │
├────────────────────────────────────────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────┼──────────┤
│ https://docs.denvrdata.com/docs/sitemap.xml                                                    │                                                                                  │ denvr    │
│ https://docs.denvrdata.com/docs/ 

So far we have a 'downloads' table and collection of saved html files in the `docs/` directory. Next, we'll use langchain to establish a vectorstore for our RAG pipeline. This is just coupling an embedding model with an `embeddings` table in the database.

In [2]:
vectorstore = yara.store(
    conn,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device='cpu',
)
conn.table('embeddings')

11/19/2024 06:16:33 - INFO - yara - Creating a vector store in the duckdb 'embeddings' table.
11/19/2024 06:16:36 - INFO - datasets - PyTorch version 2.2.2a0+gitb5d0b9b available.
11/19/2024 06:16:36 - INFO - datasets - Duckdb version 1.1.3 available.
11/19/2024 06:16:36 - INFO - sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


┌─────────┬─────────┬───────────┬──────────┐
│   id    │  text   │ embedding │ metadata │
│ varchar │ varchar │  float[]  │ varchar  │
├─────────┴─────────┴───────────┴──────────┤
│                  0 rows                  │
└──────────────────────────────────────────┘

Now we can populate that table by chunking our documents and running those chunks through the specified embedding model. For chunking, we're just using the `RecursiveCharacterTextSplitter` where `chunk_size=1500` and `chunk_overlap=100`, but langchain includes several options for this.

In [3]:
yara.populate(vectorstore, path="docs")
conn.table('embeddings')

11/19/2024 06:16:41 - INFO - yara - Chunked embeddings for overview__getting-started__launch-a-virtual-machine.json
11/19/2024 06:16:43 - INFO - yara - Chunked embeddings for additional-information__faqs__ip-access-restrictions.json
11/19/2024 06:16:43 - INFO - yara - Chunked embeddings for additional-information__faqs__do-you-support-kubernetes.json
11/19/2024 06:16:43 - INFO - yara - Chunked embeddings for api-reference__authentication.json
11/19/2024 06:16:43 - INFO - yara - Chunked embeddings for additional-information__faqs__what-ports-are-publicly-accessible.json
11/19/2024 06:16:43 - INFO - yara - Chunked embeddings for overview__whats-new.json
11/19/2024 06:16:44 - INFO - yara - Chunked embeddings for platform__networking.json
11/19/2024 06:16:44 - INFO - yara - Chunked embeddings for platform__billing.json
11/19/2024 06:16:44 - INFO - yara - Chunked embeddings for api-reference__clusters.json
11/19/2024 06:16:45 - INFO - yara - Chunked embeddings for api-reference__vpcs.json
1

┌──────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Okay, before we move on. Let's make sure our `embeddings` table is searchable from langchain.

In [4]:
query = "What network bandwidth does Denvr Dataworks offer?"
print(vectorstore.similarity_search(query)[0].page_content)

What is the network bandwidth? | Denvr Cloud DocsDenvr Cloud DocsMoreSearchCtrl +KWelcome to Denvr Cloud Docs!OVERVIEWGetting startedLaunch a virtual machineData centersShared responsibility modelTechnical supportWhat's new!PLATFORMDashboardApplicationsVirtual machinesBare metalFile systemsNetworkingUser managementBillingAPI ReferenceAuthenticationClustersVirtual machinesBare metalVPCsAdditional InformationFAQsDesktop vs data center GPUsDifferences of bare metal and virtual machinesGPU monitoringUsing Github with SSH keysIP access restrictionsData persistence and recoveryDo you support Kubernetes?Installing GPU driversWhat is the network bandwidth?What ports are publicly accessible?PoliciesTerms of ServicePrivacy PolicyMaintenance policyPowered by GitBookOn this pageMSC1 (Calgary, Canada)HOU1 (Houston, USA)What is the network bandwidth?Internet bandwidth depends on many factors including ISP speed, use of multiple connections, serving capacity, and receiving bandwidth.Denvr Cloud does 

Now we'll move on to defining our chat LLM model. Feel free to take a look at the gaudi.py file for more info, but we'll just be loading a relatively simple 7B parameter Intel model utilizing a single HPU.

In [5]:
base_model = yara.llm(
    model_name='Intel/neural-chat-7b-v3-3',
    device='hpu',
    bf16=True,
    max_new_tokens=1024,
    max_input_tokens=2048,
    batch_size=1,
    temperature=0.5,
    top_p=0.95,
    use_kv_cache=True,
    use_hpu_graphs=True,
    do_sample=True,
)
base_model.invoke("What is a Large Language Model?")

[WARNING|utils.py:212] 2024-11-19 06:17:09,266 >> optimum-habana v1.14.1 has been validated for SynapseAI v1.18.0 but habana-frameworks v1.16.2.2 was found, this could lead to undefined behavior!
[WARNING|utils.py:225] 2024-11-19 06:17:10,196 >> optimum-habana v1.14.1 has been validated for SynapseAI v1.18.0 but the driver version is v1.16.2, this could lead to undefined behavior!
11/19/2024 06:17:10 - INFO - root - Constructing HuggingFacePipeline for the GaudiTextGenerationPipeline
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

11/19/2024 06:17:12 - INFO - yara - Single-device run.
`LlamaRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

============================= HABANA PT BRIDGE CONFIGURATION =========================== 
 PT_HPU_LAZY_MODE = 1
 PT_RECIPE_CACHE_PATH = 
 PT_CACHE_FOLDER_DELETE = 0
 PT_HPU_RECIPE_CACHE_CONFIG = 
 PT_HPU_MAX_COMPOUND_OP_SIZE = 9223372036854775807
 PT_HPU_LAZY_ACC_PAR_MODE = 1
 PT_HPU_ENABLE_REFINE_DYNAMIC_SHAPES = 0
---------------------------: System Configuration :---------------------------
Num CPU Cores : 152
CPU RAM       : 970098880 KB
------------------------------------------------------------------------------
11/19/2024 06:17:21 - INFO - yara - GaudiMistralForCausalLM(
  (model): GaudiMistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x GaudiMistralDecoderLayer(
        (self_attn): GaudiMistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False

"What is a Large Language Model?\n\nA large language model is an artificial intelligence system trained on a vast amount of text data, allowing it to generate human-like text and understand natural language. These models can perform various tasks, such as text completion, translation, question answering, and summarization. Some well-known large language models include GPT-3, BERT, and T5.\n\nHow do Large Language Models work?\n\nLarge language models work by using deep learning algorithms and neural networks to analyze and process large amounts of text data. They are trained on unsupervised learning, which means they learn from unlabeled data without explicit guidance. The models continuously learn and update their knowledge as they process new data, making them more accurate and versatile.\n\nWhat are the benefits of Large Language Models?\n\nLarge language models offer several advantages, including:\n\n1. Natural Language Understanding: They can understand and process complex and var

Okay, now we're ready to build our RAG pipeline, using a custom prompt template for incorporating the context documents.

In [6]:
prompt = """
--- Instructions ---
You are a friendly AI assistant for onboarding new Denvr Dataworks employees. 
Use a conversational tone and provide helpful and informative responses, utilizing external knowledge when possible.

"**Step 1: Parse Context Information** "
Extract and utilize relevant knowledge from the provided context.
**Step 2: Analyze User Query**
Carefully read and comprehend the user's query, pinpointing the key concepts, entities, and intent behind the question.
**Step 3: Determine Response**
If the answer to the user's query can be directly inferred from the context information, provide a concise and accurate response in the same language as the user's query.
**Step 4: Handle Uncertainty**
If you don't know the answer, simply state that you don't know. If the answer is not clear, ask the user for clarification to ensure an accurate response.
**Step 5: Respond in User's Language**
Maintain consistency by ensuring the response is in the same language as the user's query.
**Step 6: Provide Response**
Generate a clear, concise, and informative response to the user's query, adhering to the guidelines outlined above.

--- Context ---
{context}

"""
rag_model = yara.rag(base_model, vectorstore, prompt)

11/19/2024 06:17:55 - INFO - root - Constructing the final RAG pipeline


To make comparing the base and rag model output a bit easier we'll use gradio `ChatInterface`.

In [7]:
response = """
--- Base Model ---
{base_resp}

--- RAG Model ---
{rag_resp_answer}

{rag_resp_refs}
"""
ui = yara.chatui(base_model, rag_model, response)
ui.launch()


/home/ubuntu/.local/lib/python3.10/site-packages/gradio/components/chatbot.py:229: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(
11/19/2024 06:18:05 - INFO - httpx - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
11/19/2024 06:18:05 - INFO - httpx - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
11/19/2024 06:18:05 - INFO - httpx - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
